# Segmentación de Otsu con validación cruzada de **K = 5** y test fijo

Aplica **Otsu** a todas las imágenes de `DATASET_FINAL2.mat` siguiendo el **mismo protocolo**
que las CNN y los transformers del TFG:

* validación cruzada **estratificada por tipo de lesión**, K = 5 pliegues
* las imágenes de índice **23–27** (con imagen registrada) son **siempre test** y nunca
  entran en entrenamiento ni en validación
* el resto de imágenes rota: en cada pliegue, 80 % train+val (reparto interno 80/20) y
  20 % test, que se suma al test fijo
* métricas reportadas como **media** entre pliegues, desglosadas en test total,
  solo rotatorias y solo fijas



### 1. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Parámetros

In [2]:
RUTA_MAT       = "/C:/Users/josem/Desktop/DATASET_FINAL2.pdf

UMBRAL_DICE   = 0.9      # solo se muestran las imágenes que superen este Dice
MAX_MOSTRAR   = 5       # tope de filas en la figura (None = todas). Evita generar
                         # una figura gigantesca si muchas imágenes pasan el filtro.
POSTPROCESO   = True     # limpieza morfológica + mayor componente conexa
RELLENAR      = True     # rellena los huecos negros dentro de la lesión
GUARDAR       = True     # guardar la figura en CARPETA_SALIDA

REDIMENSIONAR = True     # 256x256, como el resto del pipeline del TFG.
IMG_SIZE      = 256      # Ponlo a False para trabajar a resolución nativa
                         # (figuras más nítidas, métricas ligeramente distintas).

# --- validación cruzada ---
N_SPLITS      = 5        # K = 5 pliegues
VAL_FRACTION  = 0.20     # 20 % del 80 % restante = 16 % del total
RANDOM_STATE  = 42       # semilla fija -> pliegues reproducibles

# --- test fijo -------------------------------------------------------------
# Imágenes con imagen registrada del dataset: SIEMPRE son test, en todos los
# pliegues, y NUNCA entran en entrenamiento ni en validación.
INDICES_TEST_FIJOS = list(range(23, 28))   # 23, 24, 25, 26, 27

# Espacio de búsqueda de la configuración de Otsu: (canal, polaridad).
#   polaridad = 1 -> la lesión es la clase de intensidad ALTA en ese canal
#   polaridad = 0 -> es la clase BAJA
CONFIGS_CANDIDATAS = [
    ("lab_a", 1),   # crominancia rojo-verde: los hemangiomas son rojizos
    ("lab_a", 0),
    ("hsv_s", 1),   # saturación
    ("gris",  0),   # intensidad: la lesión suele ser más oscura que la piel
    ("gris",  1),
    ("rgb_r", 1),   # canal rojo
    ("inv_g", 1),   # verde invertido, realza lo rojizo
]

### 3. Importaciones

In [3]:
import os
import h5py
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import StratifiedKFold, train_test_split
from collections import Counter

### 4. Lectura del dataset

Se cargan **todas** las entradas (no una selección), porque la validación cruzada necesita el
conjunto completo.

In [4]:
f  = h5py.File(RUTA_MAT, "r")
DS = f["DATASET_UNIDO"]

def leer_entrada(i):
    c = [f[r] for r in f[DS[i, 0]][()].ravel()]
    nombre = "".join(chr(x) for x in c[0][()].ravel())
    tipo   = "".join(chr(x) for x in c[2][()].ravel())
    mask   = (c[1][()].T > 0).astype(np.uint8)                          # (H, W) binaria
    img    = (np.transpose(c[3][()], (2, 1, 0)) * 255).clip(0, 255).astype(np.uint8)
    return nombre, tipo, img, mask


N = DS.shape[0]
nombres, tipos, imagenes, mascaras = [], [], [], []

for i in range(N):
    nombre, tipo, img, gt = leer_entrada(i)
    if REDIMENSIONAR:
        # área para la imagen, vecino más próximo para la máscara,
        # que así conserva su carácter estrictamente binario
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        gt  = cv2.resize(gt,  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    nombres.append(nombre)
    tipos.append(tipo)
    imagenes.append(img)
    mascaras.append((gt > 0).astype(np.uint8))

etiquetas = np.array(tipos)

### 5. Segmentación de Otsu

In [5]:
def extraer_canal(img, canal):
    """Devuelve el canal escalar uint8 sobre el que se umbraliza."""
    if canal == "gris":
        return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    if canal == "lab_a":
        return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:, :, 1]
    if canal == "hsv_s":
        return cv2.cvtColor(img, cv2.COLOR_RGB2HSV)[:, :, 1]
    if canal == "rgb_r":
        return img[:, :, 0]
    if canal == "inv_g":
        return 255 - img[:, :, 1]
    raise ValueError(f"canal desconocido: {canal}")


def segmentar_otsu(img, canal="lab_a", polaridad=1, postproceso=True, rellenar=True):
    """Otsu selecciona automáticamente el umbral que maximiza la varianza entre las
    dos clases del histograma (equivalentemente, minimiza la varianza intraclase)."""
    ch = cv2.GaussianBlur(extraer_canal(img, canal), (5, 5), 0)

    flag = cv2.THRESH_BINARY if polaridad == 1 else cv2.THRESH_BINARY_INV
    _, seg = cv2.threshold(ch, 0, 1, flag + cv2.THRESH_OTSU)
    seg = seg.astype(np.uint8)

    if postproceso:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        seg = cv2.morphologyEx(seg, cv2.MORPH_OPEN, k)
        seg = cv2.morphologyEx(seg, cv2.MORPH_CLOSE, k)
        n, lbl = cv2.connectedComponents(seg)
        if n > 2:                       # conserva la mayor componente conexa
            areas = [(lbl == j).sum() for j in range(1, n)]
            seg = (lbl == 1 + int(np.argmax(areas))).astype(np.uint8)

    if rellenar:
        seg = binary_fill_holes(seg).astype(np.uint8)   # tapa huecos interiores

    return seg.astype(np.uint8)

### 6. Métricas y dibujo de contornos

In [24]:
EPS = 1e-8
P = 1.2
def matriz_confusion(pred, gt):
    """Recuento de VP, VN, FP y FN a nivel de píxel."""
    p, g = pred > 0, gt > 0
    return dict(vp=int(( p &  g).sum()), vn=int((~p & ~g).sum()),
                fp=int(( p & ~g).sum()), fn=int((~p &  g).sum()))

def suma_confusion(a, b):
    return {k: a[k] + b[k] for k in a}

def dice_de_confusion(c):
    return (2 * c["vp"] + EPS) / (2 * c["vp"] + c["fp"] + c["fn"] + EPS)

def metricas_de_confusion(c):
    return {
        "exactitud":    (c["vp"] + c["vn"]) / (c["vp"] + c["vn"] + c["fp"] + c["fn"] + EPS),
        "precision":    (c["vp"] + EPS) / (c["vp"] + c["fp"]+ EPS),
        "sensibilidad":  (c["vp"] + EPS) / (c["vp"] + c["fn"] + EPS),
        "especificidad": (c["vn"] + EPS) / (c["vn"] + c["fp"] + EPS),
    }

def dice(a, b):
    return dice_de_confusion(matriz_confusion(a, b))


def evaluar(indices, canal, polaridad):
    """Evalúa una configuración de Otsu sobre un subconjunto de índices.

    Distingue las dos formas de agregar, tal y como se explica en §6.2 de la memoria:
      * Dice -> se calcula por imagen y se promedia (macro). Cada lesión pesa igual.
      * F1   -> se acumulan VP, FP y FN de todas las imágenes y se calcula una sola
                vez sobre el total (micro). Cada píxel pesa igual, así que las
                lesiones grandes influyen más.
    Por eso Dice y F1 comparten fórmula pero no valor."""
    dices, total = [], dict(vp=0, vn=0, fp=0, fn=0)
    for i in indices:
        seg = segmentar_otsu(imagenes[i], canal, polaridad, POSTPROCESO, RELLENAR)
        c = matriz_confusion(seg, mascaras[i])
        dices.append(dice_de_confusion(c))
        total = suma_confusion(total, c)
    res = metricas_de_confusion(total)
    res["dice"] = float(np.mean(dices))   # macro, por imagen
    res["f1"]   = dice_de_confusion(total)  # micro, global
    return res


def elegir_configuracion(indices_ajuste):
    """Elige la configuración que maximiza el Dice en entrenamiento + validación.
    Ninguna imagen de test interviene en esta decisión."""
    mejor, mejor_dice = None, -1.0
    for canal, polaridad in CONFIGS_CANDIDATAS:
        d = evaluar(indices_ajuste, canal, polaridad)["dice"]
        if d > mejor_dice:
            mejor, mejor_dice = (canal, polaridad), d
    return mejor, mejor_dice


def dibujar_contornos(img, seg, gt, grosor=None):
    o = img.copy()
    if grosor is None:
        grosor = max(2, int(round(max(img.shape[:2]) / 300)))   # grosor según tamaño
    cont_gt,  _ = cv2.findContours(gt.astype(np.uint8),  cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cont_seg, _ = cv2.findContours(seg.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(o, cont_gt,  -1, (255, 0, 0), grosor)   # ground truth -> rojo
    cv2.drawContours(o, cont_seg, -1, (0, 255, 0), grosor)   # Otsu         -> verde
    return o

### 7. Validación cruzada estratificada de 5 pliegues con test fijo

Las imágenes de índice **23–27** (las que tienen imagen registrada) se reservan como
**test en todos los pliegues** y **nunca** participan en la elección de la configuración
de Otsu (ni en entrenamiento ni en validación).

El resto del dataset se reparte con `StratifiedKFold`, de modo que en cada pliegue:

* **test** = pliegue rotatorio + las 5 imágenes fijas
* **train / val** = el 80 % restante de las imágenes rotatorias (80/20 interno)

Como las fijas se evalúan K veces (una por pliegue, con la configuración de cada uno),
su Dice fuera de pliegue se reporta como **media entre los K pliegues**.

In [25]:
# --- conjunto de test fijo -------------------------------------------------
FIJOS = np.array(sorted(set(INDICES_TEST_FIJOS)), dtype=int)
assert FIJOS.min() >= 0 and FIJOS.max() < N, "INDICES_TEST_FIJOS fuera de rango"

# El resto del dataset es lo único que rota entre entrenamiento, validación y test
RESTO = np.array([i for i in range(N) if i not in set(FIJOS.tolist())], dtype=int)

print(f"Test fijo ({len(FIJOS)} imágenes, siempre en test, nunca en train/val):")
for i in FIJOS:
    print(f"   idx {i:>3}  {nombres[i]}")
print(f"Imágenes que rotan en la validación cruzada: {len(RESTO)}\n")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

resultados_pliegue       = []   # test completo  = fold rotatorio + fijas
resultados_pliegue_rot   = []   # solo la parte rotatoria del test
resultados_pliegue_fijos = []   # solo las imágenes fijas
configs_elegidas         = []

# predicción fuera de pliegue para cada imagen
seg_oof    = [None] * N
dice_oof   = np.zeros(N)
pliegue_de = np.zeros(N, dtype=int)
dice_fijos_por_pliegue = {i: [] for i in FIJOS}   # las fijas se predicen K veces

print(f"Validación cruzada estratificada de {N_SPLITS} pliegues\n")

for pliegue, (pos_trval, pos_test) in enumerate(
        skf.split(np.zeros(len(RESTO)), etiquetas[RESTO]), 1):

    idx_trval    = RESTO[pos_trval]
    idx_test_rot = RESTO[pos_test]
    # El test de cada pliegue = parte rotatoria + las imágenes fijas
    idx_test     = np.concatenate([idx_test_rot, FIJOS])

    # Separación interna entrenamiento / validación
    try:
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION,
            stratify=etiquetas[idx_trval], random_state=RANDOM_STATE)
    except ValueError:
        # alguna clase tiene muy pocas muestras para estratificar el subreparto
        idx_train, idx_val = train_test_split(
            idx_trval, test_size=VAL_FRACTION, random_state=RANDOM_STATE)

    # La configuración se decide sobre entrenamiento + validación...
    (canal, polaridad), dice_ajuste = elegir_configuracion(
        np.concatenate([idx_train, idx_val]))
    # ...y se aplica al pliegue de test, no visto en la selección.
    res       = evaluar(idx_test,     canal, polaridad)
    res_rot   = evaluar(idx_test_rot, canal, polaridad)
    res_fijos = evaluar(FIJOS,        canal, polaridad)

    for i in idx_test:
        s = segmentar_otsu(imagenes[i], canal, polaridad, POSTPROCESO, RELLENAR)
        d = dice(s, mascaras[i])
        seg_oof[i]    = s          # para las fijas queda la del último pliegue
        dice_oof[i]   = d
        pliegue_de[i] = pliegue
        if i in dice_fijos_por_pliegue:
            dice_fijos_por_pliegue[i].append(d)

    resultados_pliegue.append(res)
    resultados_pliegue_rot.append(res_rot)
    resultados_pliegue_fijos.append(res_fijos)
    configs_elegidas.append((canal, polaridad))



# El Dice fuera de pliegue de las imágenes fijas se promedia entre los K pliegues
for i in FIJOS:
    dice_oof[i] = float(np.mean(dice_fijos_por_pliegue[i]))

Test fijo (5 imágenes, siempre en test, nunca en train/val):
   idx  23  imagen20.jpg
   idx  24  imagen21.jpg
   idx  25  imagen24.jpg
   idx  26  imagen29.jpg
   idx  27  imagen3.jpg
Imágenes que rotan en la validación cruzada: 125

Validación cruzada estratificada de 5 pliegues



### 8. Resumen:

In [26]:
CLAVES  = ["exactitud", "sensibilidad", "especificidad","precision", "dice", "f1"]
NOMBRES = {"exactitud": "Exactitud", "precision": "Precisión",
           "sensibilidad": "Sensibilidad", "especificidad": "Especificidad",
           "dice": "Dice", "f1": "F1"}

def resumir(lista):
    return {k: (float(np.mean([r[k] for r in lista])),
                float(np.std ([r[k] for r in lista]))) for k in CLAVES}

resumen       = resumir(resultados_pliegue)
resumen_rot   = resumir(resultados_pliegue_rot)
resumen_fijos = resumir(resultados_pliegue_fijos)

print("=" * 72)
print(f"Otsu — validación cruzada de {N_SPLITS} pliegues ")
print("=" * 72)
print(f"{'Métrica':<16}{'Test total':>14}")
print("-" * 72)
for k in CLAVES:
    print(f"{NOMBRES[k]:<16}{resumen[k][0]:>14.4f}")
print("-" * 72)



Otsu — validación cruzada de 5 pliegues 
Métrica             Test total
------------------------------------------------------------------------
Exactitud               0.8802
Sensibilidad            0.9000
Especificidad           0.9266
Precisión               0.8745
Dice                    0.8191
F1                      0.7855
------------------------------------------------------------------------
